# Dataset Quantitative Analysis
**Zero-Shot Nutrient and Quantity Association using OCR-Guided Semantic Graph Matching**  
Moustafa Hamada | M.Sc. AI & Data Science | THD + USB

This notebook performs a full quantitative analysis of the JSON annotation dataset.  
All mass quantities are normalised to **grams**. Energy rows (kJ, kcal) are excluded from quantity graphs.

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Seaborn theme ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE   = sns.color_palette('muted')
FIG_DPI   = 150
SAVE_FIGS = True          # set False to skip saving
FIG_DIR   = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

# ── Paths ─────────────────────────────────────────────────────────────────────
ANNOTATIONS_DIR = Path('data/annotations')   # adjust if needed

print(f'Annotations directory : {ANNOTATIONS_DIR.resolve()}')
print(f'Exists                : {ANNOTATIONS_DIR.exists()}')

## 1. Load Dataset

In [ ]:
records = []

for jf in sorted(ANNOTATIONS_DIR.glob('*.json')):
    with open(jf, encoding='utf-8') as f:
        data = json.load(f)
    image_id = data.get('image_id', jf.stem + '.jpeg')
    for n in data.get('nutrients', []):
        records.append({
            'image_id':    image_id,
            'nutrient':    n.get('nutrient',    '').strip(),
            'quantity':    n.get('quantity',    None),
            'unit':        n.get('unit',        '').strip().lower(),
            'context':     n.get('context',     '').strip(),
            'nrv_percent': n.get('nrv_percent', None),
            'serving_size':n.get('serving_size',None),
        })

df = pd.DataFrame(records)

# ── Type coercions ────────────────────────────────────────────────────────────
df['quantity']    = pd.to_numeric(df['quantity'],    errors='coerce')
df['nrv_percent'] = pd.to_numeric(df['nrv_percent'], errors='coerce')

print(f'Total tuples  : {len(df)}')
print(f'Images        : {df["image_id"].nunique()}')
print(f'\nColumn dtypes:')
print(df.dtypes)
df.head()

## 2. Dataset Overview

In [ ]:
print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'  Total annotation tuples : {len(df):>6}')
print(f'  Total images            : {df["image_id"].nunique():>6}')
print(f'  Unique nutrients        : {df["nutrient"].nunique():>6}')
print(f'  Unique units            : {df["unit"].nunique():>6}')
print(f'  Unique contexts         : {df["context"].nunique():>6}')
print(f'  Tuples with NRV%        : {df["nrv_percent"].notna().sum():>6}')
print(f'  Tuples with serving size: {df["serving_size"].notna().sum():>6}')
print('=' * 55)

print('\nDescriptive stats — quantity column (raw, before unit normalisation):')
print(df['quantity'].describe().round(4))

## 3. Unit Normalisation to Grams
Conversion factors applied:  
- `µg / µg / mcg` → ÷ 1,000,000  
- `mg` → ÷ 1,000  
- `g` → ×1 (identity)  
- `kg` → ×1,000  
- `kj / kcal` → **excluded** from quantity graphs  

In [ ]:
# ── Unit normalisation map ────────────────────────────────────────────────────
UNIT_TO_GRAMS = {
    'µg':   1e-6,  'ug':   1e-6,  'mcg':  1e-6,  'μg':   1e-6,
    'mg':   1e-3,
    'g':    1.0,
    'kg':   1e3,
}
ENERGY_UNITS = {'kj', 'kcal', 'cal', 'kbe'}

def to_grams(row):
    unit = str(row['unit']).strip().lower()
    qty  = row['quantity']
    if pd.isna(qty):
        return np.nan
    factor = UNIT_TO_GRAMS.get(unit)
    if factor is None:
        return np.nan
    return float(qty) * factor

df['is_energy']    = df['unit'].str.lower().isin(ENERGY_UNITS)
df['unit_known']   = df['unit'].str.lower().isin(set(UNIT_TO_GRAMS) | ENERGY_UNITS)
df['qty_grams']    = df.apply(to_grams, axis=1)

# ── Mass-only subset (energy excluded) ───────────────────────────────────────
df_mass = df[~df['is_energy'] & df['qty_grams'].notna()].copy()

print(f'Tuples with mass unit       : {len(df_mass)}')
print(f'Tuples with energy unit     : {df["is_energy"].sum()}')
print(f'Tuples with unknown unit    : {(~df["unit_known"]).sum()}')
print(f'\nDescriptive stats — qty_grams (mass only):')
print(df_mass['qty_grams'].describe().round(8))

## 4. Tuples per Image

In [ ]:
tuples_per_image = df.groupby('image_id').size().sort_values(ascending=False)

print('Tuples per image — descriptive statistics:')
stats = tuples_per_image.describe()
print(stats.round(2))
print(f'  Median : {tuples_per_image.median():.1f}')
print(f'  Mode   : {tuples_per_image.mode().values}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — sorted descending
ax = axes[0]
tuples_per_image.plot(kind='bar', ax=ax, color=PALETTE[0], edgecolor='white', linewidth=0.5)
ax.axhline(tuples_per_image.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean={tuples_per_image.mean():.1f}')
ax.axhline(tuples_per_image.median(), color='orange', linestyle=':',  linewidth=1.5, label=f'Median={tuples_per_image.median():.1f}')
ax.set_title('Tuples per Image (sorted)', fontweight='bold')
ax.set_xlabel('Image ID')
ax.set_ylabel('Number of tuples')
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.legend()

# Histogram
ax2 = axes[1]
sns.histplot(tuples_per_image, bins=15, kde=True, ax=ax2, color=PALETTE[0])
ax2.axvline(tuples_per_image.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean={tuples_per_image.mean():.1f}')
ax2.axvline(tuples_per_image.median(), color='orange', linestyle=':',  linewidth=1.5, label=f'Median={tuples_per_image.median():.1f}')
ax2.set_title('Distribution of Tuples per Image', fontweight='bold')
ax2.set_xlabel('Number of tuples')
ax2.set_ylabel('Number of images')
ax2.legend()

plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'tuples_per_image.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 5. Context Distribution

In [ ]:
ctx_counts = df['context'].value_counts()

print('Context distribution:')
print(ctx_counts.to_frame('count').assign(pct=lambda x: (x['count']/len(df)*100).round(2)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
ax = axes[0]
ctx_counts.plot(kind='bar', ax=ax, color=PALETTE[2], edgecolor='white')
ax.set_title('Context Label Frequency', fontweight='bold')
ax.set_xlabel('Context')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    ctx_counts.values, labels=ctx_counts.index,
    autopct='%1.1f%%', colors=sns.color_palette('pastel'),
    startangle=140, pctdistance=0.82,
)
for t in autotexts: t.set_fontsize(10)
ax2.set_title('Context Label Proportions', fontweight='bold')

plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'context_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 6. Unit Distribution

In [ ]:
unit_counts = df['unit'].value_counts()

print('Unit distribution:')
print(unit_counts.to_frame('count').assign(pct=lambda x: (x['count']/len(df)*100).round(2)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# All units bar chart
ax = axes[0]
unit_counts.plot(kind='bar', ax=ax, color=PALETTE[3], edgecolor='white')
ax.set_title('Unit Frequency (all)', fontweight='bold')
ax.set_xlabel('Unit')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)

# Mass vs energy split
ax2 = axes[1]
split = pd.Series({
    'Mass (g/mg/µg/kg)': (~df['is_energy'] & df['unit_known']).sum(),
    'Energy (kJ/kcal)':  df['is_energy'].sum(),
    'Unknown unit':      (~df['unit_known']).sum(),
})
split.plot(kind='bar', ax=ax2, color=[PALETTE[0], PALETTE[1], PALETTE[5]], edgecolor='white')
ax2.set_title('Mass vs Energy vs Unknown', fontweight='bold')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=20)
for p in ax2.patches:
    ax2.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                 ha='center', va='bottom', fontsize=11)

plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'unit_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 7. Quantity Distribution (in grams, energy excluded)

In [ ]:
qty_g = df_mass['qty_grams']

print('Quantity descriptive statistics (grams, energy excluded):')
print(f'  Count   : {qty_g.count()}')
print(f'  Mean    : {qty_g.mean():.6f} g')
print(f'  Median  : {qty_g.median():.6f} g')
print(f'  Std     : {qty_g.std():.6f} g')
print(f'  Min     : {qty_g.min():.8f} g')
print(f'  Max     : {qty_g.max():.4f} g')
print(f'  Q25     : {qty_g.quantile(0.25):.6f} g')
print(f'  Q75     : {qty_g.quantile(0.75):.6f} g')
print(f'  IQR     : {qty_g.quantile(0.75) - qty_g.quantile(0.25):.6f} g')
print(f'  Skewness: {qty_g.skew():.4f}')
print(f'  Kurtosis: {qty_g.kurtosis():.4f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Full range histogram (log scale x) ───────────────────────────────────────
ax = axes[0, 0]
log_vals = np.log10(qty_g[qty_g > 0])
sns.histplot(log_vals, bins=40, kde=True, ax=ax, color=PALETTE[0])
ax.axvline(np.log10(qty_g.median()), color='orange', linestyle='--', linewidth=1.5,
           label=f'Median={qty_g.median():.4f}g')
ax.axvline(np.log10(qty_g.mean()),   color='red',    linestyle='-.',  linewidth=1.5,
           label=f'Mean={qty_g.mean():.4f}g')
ax.set_xlabel('log₁₀(quantity in grams)')
ax.set_ylabel('Count')
ax.set_title('Quantity Distribution (log scale)', fontweight='bold')
ax.legend(fontsize=9)

# Annotate tick labels with actual gram values
ticks = ax.get_xticks()
ax.set_xticklabels([f'{10**t:.4g}g' for t in ticks], rotation=30, fontsize=8)

# ── Zoomed: 0–1g range ───────────────────────────────────────────────────────
ax2 = axes[0, 1]
sub = qty_g[qty_g <= 1.0]
sns.histplot(sub, bins=40, kde=True, ax=ax2, color=PALETTE[1])
ax2.axvline(sub.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median={sub.median():.4f}g')
ax2.axvline(sub.mean(),   color='red',    linestyle='-.',  linewidth=1.5, label=f'Mean={sub.mean():.4f}g')
ax2.set_title('Quantity Distribution (0 – 1 g range)', fontweight='bold')
ax2.set_xlabel('Quantity (grams)')
ax2.set_ylabel('Count')
ax2.legend(fontsize=9)

# ── Box plot by context ───────────────────────────────────────────────────────
ax3 = axes[1, 0]
plot_data = df_mass[['context', 'qty_grams']].copy()
plot_data['log_qty'] = np.log10(plot_data['qty_grams'].clip(lower=1e-9))
sns.boxplot(data=plot_data, x='context', y='log_qty', ax=ax3,
            palette='muted', flierprops=dict(marker='o', markersize=3, alpha=0.5))
ax3.set_title('Quantity (log₁₀ g) by Context', fontweight='bold')
ax3.set_xlabel('Context')
ax3.set_ylabel('log₁₀(quantity in grams)')
ax3.tick_params(axis='x', rotation=20)

# ── Violin plot by original unit ─────────────────────────────────────────────
ax4 = axes[1, 1]
top_units = df_mass['unit'].value_counts().head(5).index.tolist()
sub_units = df_mass[df_mass['unit'].isin(top_units)].copy()
sub_units['log_qty'] = np.log10(sub_units['qty_grams'].clip(lower=1e-9))
sns.violinplot(data=sub_units, x='unit', y='log_qty', ax=ax4,
               palette='muted', inner='box', cut=0)
ax4.set_title('Quantity (log₁₀ g) by Unit (top 5)', fontweight='bold')
ax4.set_xlabel('Original unit')
ax4.set_ylabel('log₁₀(quantity in grams)')

plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'quantity_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 8. Nutrient Name Frequency

In [ ]:
TOP_N = 30
nutrient_counts = df['nutrient'].value_counts().head(TOP_N)

print(f'Top {TOP_N} most frequent nutrients:')
print(nutrient_counts.to_frame('count').assign(pct=lambda x: (x['count']/len(df)*100).round(2)))

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    nutrient_counts.index[::-1],
    nutrient_counts.values[::-1],
    color=sns.color_palette('Blues_r', len(nutrient_counts)),
    edgecolor='white'
)
ax.axvline(nutrient_counts.mean(),   color='red',    linestyle='--', linewidth=1.5,
           label=f'Mean={nutrient_counts.mean():.1f}')
ax.axvline(nutrient_counts.median(), color='orange', linestyle=':',  linewidth=1.5,
           label=f'Median={nutrient_counts.median():.1f}')
ax.set_xlabel('Frequency (number of images)')
ax.set_title(f'Top {TOP_N} Nutrient Names by Frequency', fontweight='bold')
ax.legend()
for bar, val in zip(bars, nutrient_counts.values[::-1]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'nutrient_frequency.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

print(f'\nTotal unique nutrient names : {df["nutrient"].nunique()}')
print(f'Nutrients appearing once    : {(df["nutrient"].value_counts() == 1).sum()}')

## 9. NRV% Distribution

In [ ]:
nrv = df['nrv_percent'].dropna()

print('NRV% descriptive statistics:')
print(f'  Tuples with NRV%  : {len(nrv)} / {len(df)} ({len(nrv)/len(df)*100:.1f}%)')
print(f'  Mean              : {nrv.mean():.2f}%')
print(f'  Median            : {nrv.median():.2f}%')
print(f'  Std               : {nrv.std():.2f}%')
print(f'  Min               : {nrv.min():.2f}%')
print(f'  Max               : {nrv.max():.2f}%')
print(f'  Q25               : {nrv.quantile(0.25):.2f}%')
print(f'  Q75               : {nrv.quantile(0.75):.2f}%')
print(f'  Skewness          : {nrv.skew():.4f}')
print(f'  Kurtosis          : {nrv.kurtosis():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sns.histplot(nrv, bins=40, kde=True, ax=ax, color=PALETTE[4])
ax.axvline(nrv.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean={nrv.mean():.1f}%')
ax.axvline(nrv.median(), color='orange', linestyle=':',  linewidth=1.5, label=f'Median={nrv.median():.1f}%')
ax.set_title('NRV% Distribution', fontweight='bold')
ax.set_xlabel('NRV (%)')
ax.set_ylabel('Count')
ax.legend()

ax2 = axes[1]
# Clipped at 500% to avoid extreme outlier distortion
nrv_clipped = nrv[nrv <= 500]
sns.boxplot(y=nrv_clipped, ax=ax2, color=PALETTE[4],
            flierprops=dict(marker='o', markersize=3, alpha=0.5))
ax2.set_title('NRV% Box Plot (clipped at 500%)', fontweight='bold')
ax2.set_ylabel('NRV (%)')

plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'nrv_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 10. Serving Size Analysis

In [ ]:
ss = df['serving_size'].dropna()
ss_counts = ss.value_counts().head(20)

print(f'Tuples with serving size : {len(ss)} / {len(df)} ({len(ss)/len(df)*100:.1f}%)')
print(f'Unique serving sizes     : {ss.nunique()}')
print(f'\nTop 20 serving sizes:')
print(ss_counts.to_frame('count').assign(pct=lambda x: (x['count']/len(ss)*100).round(2)))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    ss_counts.index[::-1],
    ss_counts.values[::-1],
    color=sns.color_palette('Greens_r', len(ss_counts)),
    edgecolor='white'
)
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Serving Sizes by Frequency', fontweight='bold')
for bar, val in zip(bars, ss_counts.values[::-1]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'serving_size_distribution.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 11. Missing Fields Analysis

In [ ]:
fields = ['nutrient', 'quantity', 'unit', 'context', 'nrv_percent', 'serving_size']

missing = pd.DataFrame({
    'field':   fields,
    'missing': [df[f].isna().sum() for f in fields],
    'present': [df[f].notna().sum() for f in fields],
})
missing['missing_pct'] = (missing['missing'] / len(df) * 100).round(2)
missing['present_pct'] = (missing['present'] / len(df) * 100).round(2)

print('Missing field analysis:')
print(missing.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Stacked bar — present vs missing
ax = axes[0]
x = np.arange(len(fields))
ax.bar(x, missing['present_pct'], label='Present', color=PALETTE[0], edgecolor='white')
ax.bar(x, missing['missing_pct'], bottom=missing['present_pct'],
       label='Missing / null', color=PALETTE[5], edgecolor='white', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(fields, rotation=20, ha='right')
ax.set_ylabel('Percentage of tuples (%)')
ax.set_title('Field Completeness', fontweight='bold')
ax.legend()
ax.set_ylim(0, 110)
for i, row in missing.iterrows():
    ax.text(i, row['present_pct'] + 1, f"{row['present_pct']:.0f}%",
            ha='center', fontsize=9)

# Absolute missing counts
ax2 = axes[1]
ax2.bar(fields, missing['missing'], color=PALETTE[5], edgecolor='white')
ax2.set_title('Missing Values (absolute count)', fontweight='bold')
ax2.set_ylabel('Number of null tuples')
ax2.tick_params(axis='x', rotation=20)
for i, val in enumerate(missing['missing']):
    ax2.text(i, val + 0.5, str(val), ha='center', fontsize=10)

plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'missing_fields.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 12. Cross-Analysis: Quantity (g) by Nutrient (top 15)

In [ ]:
TOP_NUTRIENTS = 15
top_nuts = df_mass['nutrient'].value_counts().head(TOP_NUTRIENTS).index.tolist()
sub = df_mass[df_mass['nutrient'].isin(top_nuts)].copy()
sub['log_qty'] = np.log10(sub['qty_grams'].clip(lower=1e-9))

# Summary table
summary = sub.groupby('nutrient')['qty_grams'].agg(
    count='count', mean='mean', median='median', std='std', min='min', max='max'
).round(6).sort_values('count', ascending=False)
print(f'Quantity (grams) summary for top {TOP_NUTRIENTS} nutrients:')
print(summary.to_string())

fig, ax = plt.subplots(figsize=(14, 7))
order = sub.groupby('nutrient')['log_qty'].median().sort_values().index
sns.boxplot(
    data=sub, x='log_qty', y='nutrient', order=order,
    palette='muted', ax=ax,
    flierprops=dict(marker='o', markersize=3, alpha=0.4)
)
ax.set_title(f'Quantity Distribution (log₁₀ g) — Top {TOP_NUTRIENTS} Nutrients', fontweight='bold')
ax.set_xlabel('log₁₀(quantity in grams)')
ax.set_ylabel('Nutrient')
plt.tight_layout()
if SAVE_FIGS: plt.savefig(FIG_DIR / 'qty_by_nutrient.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 13. Summary Statistics Table

In [ ]:
print('=' * 65)
print('  FULL DATASET SUMMARY')
print('=' * 65)
print(f'  Total annotation tuples        : {len(df)}')
print(f'  Total images annotated         : {df["image_id"].nunique()}')
print(f'  Avg tuples per image           : {len(df)/df["image_id"].nunique():.2f}')
print(f'  Median tuples per image        : {df.groupby("image_id").size().median():.1f}')
print(f'  Unique nutrient names          : {df["nutrient"].nunique()}')
print(f'  Unique units                   : {df["unit"].nunique()}')
print(f'  Tuples — mass unit             : {len(df_mass)} ({len(df_mass)/len(df)*100:.1f}%)')
print(f'  Tuples — energy unit           : {df["is_energy"].sum()} ({df["is_energy"].sum()/len(df)*100:.1f}%)')
print(f'  Tuples — with NRV%             : {df["nrv_percent"].notna().sum()} ({df["nrv_percent"].notna().sum()/len(df)*100:.1f}%)')
print(f'  Tuples — with serving size     : {df["serving_size"].notna().sum()} ({df["serving_size"].notna().sum()/len(df)*100:.1f}%)')
print('-' * 65)
print(f'  Quantity (grams, mass only):')
print(f'    Mean   : {df_mass["qty_grams"].mean():.6f} g')
print(f'    Median : {df_mass["qty_grams"].median():.6f} g')
print(f'    Std    : {df_mass["qty_grams"].std():.6f} g')
print(f'    Min    : {df_mass["qty_grams"].min():.8f} g')
print(f'    Max    : {df_mass["qty_grams"].max():.4f} g')
print('-' * 65)
print(f'  NRV% (where present):')
print(f'    Mean   : {df["nrv_percent"].mean():.2f}%')
print(f'    Median : {df["nrv_percent"].median():.2f}%')
print(f'    Std    : {df["nrv_percent"].std():.2f}%')
print(f'    Max    : {df["nrv_percent"].max():.2f}%')
print('=' * 65)